# Tensor columns in nested-pandas

A `TensorExtensionArray` holds one fixed-shape numpy array per row, stored as a pyarrow `fixed_shape_tensor` array. This is a quick tour: building them, what they look like, reading them back out, serialization, and what is not there yet.

In [1]:
import pickle
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import nested_pandas as npd
from nested_pandas import TensorDtype
from nested_pandas.tensors import TensorExtensionArray

## Creating

The main entry point is `from_stack`: a numpy array of shape `(n, *tensor_shape)`, one tensor per row along the first axis. For C-contiguous input this wraps the numpy memory without copying.

In [2]:
stack = np.arange(24.0).reshape(4, 2, 3)   # four 2x3 tensors
arr = TensorExtensionArray.from_stack(stack)
arr

<TensorExtensionArray>
[      array([[0., 1., 2.],
       [3., 4., 5.]]),
 array([[ 6.,  7.,  8.],
       [ 9., 10., 11.]]),
 array([[12., 13., 14.],
       [15., 16., 17.]]),
 array([[18., 19., 20.],
       [21., 22., 23.]])]
Length: 4, dtype: tensor[double, (2, 3)]

In [3]:
arr.dtype, len(arr), arr.dtype.shape, arr.dtype.value_type

(tensor[double, (2, 3)], 4, (2, 3), DataType(double))

`from_sequence` takes a list of individual tensors, with `None` or `pd.NA` for missing rows. The dtype is inferred from the first present element, or can be given.

In [4]:
TensorExtensionArray.from_sequence([np.zeros((2, 3)), None, np.ones((2, 3))])

<TensorExtensionArray>
[array([[0., 0., 0.],
       [0., 0., 0.]]),
                                       <NA>,
 array([[1., 1., 1.],
       [1., 1., 1.]])]
Length: 3, dtype: tensor[double, (2, 3)]

The dtype has a string spelling, so the usual pandas constructors work too. `pd.array`, `pd.Series(..., dtype=...)` and `astype` all accept it.

In [5]:
dtype = TensorDtype(pa.fixed_shape_tensor(pa.float32(), [2, 3]))
print(dtype, "==", pd.api.types.pandas_dtype("tensor[float, (2, 3)]"))
pd.array(list(stack), dtype="tensor[float, (2, 3)]")

tensor[float, (2, 3)] == tensor[float, (2, 3)]


<TensorExtensionArray>
[      array([[0., 1., 2.],
       [3., 4., 5.]], dtype=float32),
 array([[ 6.,  7.,  8.],
       [ 9., 10., 11.]], dtype=float32),
 array([[12., 13., 14.],
       [15., 16., 17.]], dtype=float32),
 array([[18., 19., 20.],
       [21., 22., 23.]], dtype=float32)]
Length: 4, dtype: tensor[float, (2, 3)]

## What they look like

Tensor columns sit in a `NestedFrame` (or a plain DataFrame) like any other column. Each row prints as a nested list, unless the tensor is large, in which case a shape and dtype descriptor is shown instead.

In [6]:
s = pd.Series(arr, name="small")
s

0          [[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]]
1        [[6.0, 7.0, 8.0], [9.0, 10.0, 11.0]]
2    [[12.0, 13.0, 14.0], [15.0, 16.0, 17.0]]
3    [[18.0, 19.0, 20.0], [21.0, 22.0, 23.0]]
Name: small, dtype: tensor[double, (2, 3)]

In [7]:
images = TensorExtensionArray.from_stack(np.random.default_rng(0).random((4, 64, 64)).astype(np.float32))
df = npd.NestedFrame({"id": [10, 11, 12, 13], "small": arr, "image": images})
df

,id,small,image
0,10,"[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]]",[64×64] float32
1,11,"[[6.0, 7.0, 8.0], [9.0, 10.0, 11.0]]",[64×64] float32
2,12,"[[12.0, 13.0, 14.0], [15.0, 16.0, 17.0]]",[64×64] float32
3,13,"[[18.0, 19.0, 20.0], [21.0, 22.0, 23.0]]",[64×64] float32


In [8]:
df.dtypes

id                         int64
small     tensor[double, (2, 3)]
image    tensor[float, (64, 64)]
dtype: object

## Getting data back out

Scalar access gives a read-only numpy view over the arrow buffers, so no data is copied. `to_stack` gives the whole column back as one `(n, *shape)` array, again as a view when there are no missing values.

In [9]:
first = df["small"].iloc[0]
first, first.flags.writeable

(array([[0., 1., 2.],
        [3., 4., 5.]]),
 False)

In [10]:
whole = df["image"].array.to_stack()
whole.shape, whole.dtype, np.shares_memory(whole, np.asarray(images.storage.chunk(0).values))

((4, 64, 64), dtype('float32'), True)

Missing rows are `pd.NA` on scalar access and are filled with `na_value` by `to_stack`. For integer tensors the fill has to be an integer, otherwise the dtype would silently widen.

In [11]:
with_missing = TensorExtensionArray.from_sequence([np.zeros((2, 3)), None, np.ones((2, 3))])
print(with_missing.isna())
print(with_missing[1])
with_missing.to_stack(na_value=-1.0)

[False  True False]
<NA>


array([[[ 0.,  0.,  0.],
        [ 0.,  0.,  0.]],

       [[-1., -1., -1.],
        [-1., -1., -1.]],

       [[ 1.,  1.,  1.],
        [ 1.,  1.,  1.]]])

In [12]:
ints = TensorExtensionArray.from_sequence([np.zeros((2, 2), dtype=np.int32), None])
try:
    ints.to_stack()
except Exception as e:
    print(type(e).__name__, e)
ints.to_stack(na_value=0).dtype

dtype('int32')

## Selecting and modifying

Everything positional works: slicing, boolean masks, `take`, `concat`, DataFrame filtering. Assignment works at the array level and for single labels on a Series. Every operation builds a new arrow array; nothing is mutated in place.

In [13]:
df[df["id"] > 11]

,id,small,image
2,12,"[[12.0, 13.0, 14.0], [15.0, 16.0, 17.0]]",[64×64] float32
3,13,"[[18.0, 19.0, 20.0], [21.0, 22.0, 23.0]]",[64×64] float32


In [14]:
arr2 = arr.copy()
arr2[0] = np.full((2, 3), 9.0)              # one tensor
arr2[[1, 2]] = [np.zeros((2, 3)), np.ones((2, 3))]   # a sequence of tensors
arr2

<TensorExtensionArray>
[      array([[9., 9., 9.],
       [9., 9., 9.]]),
       array([[0., 0., 0.],
       [0., 0., 0.]]),
       array([[1., 1., 1.],
       [1., 1., 1.]]),
 array([[18., 19., 20.],
       [21., 22., 23.]])]
Length: 4, dtype: tensor[double, (2, 3)]

In [15]:
s2 = s.copy()
s2[3] = np.full((2, 3), -1.0)
s2

0          [[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]]
1        [[6.0, 7.0, 8.0], [9.0, 10.0, 11.0]]
2    [[12.0, 13.0, 14.0], [15.0, 16.0, 17.0]]
3    [[-1.0, -1.0, -1.0], [-1.0, -1.0, -1.0]]
Name: small, dtype: tensor[double, (2, 3)]

In [16]:
pd.concat([s, s2], ignore_index=True).array.num_chunks

2

`astype` casts the element type, or reshapes to another shape with the same number of elements.

In [17]:
arr.astype("tensor[float, (2, 3)]").dtype, arr.astype("tensor[double, (6,)]")[0]

(tensor[float, (2, 3)], array([0., 1., 2., 3., 4., 5.]))

## Arrow interop

The array is a thin wrapper over a pyarrow `fixed_shape_tensor` chunked array, and converts to and from it without copying. `pa.Table.from_pandas` preserves the tensor type.

In [18]:
pa_array = arr.pa_array
pa_array.type

FixedShapeTensorType(extension<arrow.fixed_shape_tensor[value_type=double, shape=[2,3]]>)

In [19]:
TensorExtensionArray(pa_array).equals(arr), TensorDtype(pa_array.type) == arr.dtype

(True, True)

In [20]:
table = pa.Table.from_pandas(df)
table.schema

id: int64
small: extension<arrow.fixed_shape_tensor[value_type=double, shape=[2,3]]>
image: extension<arrow.fixed_shape_tensor[value_type=float, shape=[64,64]]>
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 608

## Round trips

Parquet, through pyarrow's tensor extension type, and pickle. `to_parquet` writes the column as a `fixed_shape_tensor`, and `nested_pandas.read_parquet` maps it straight back to the tensor dtype, where plain `pandas.read_parquet` would give an `ArrowDtype` of the extension type.

In [21]:
tmpdir = tempfile.TemporaryDirectory()
path = Path(tmpdir.name) / "tensors.parquet"
df.to_parquet(path)
back = npd.read_parquet(path)
back.dtypes

id                int64[pyarrow]
small     tensor[double, (2, 3)]
image    tensor[float, (64, 64)]
dtype: object

In [22]:
np.array_equal(back["image"].array.to_stack(), whole), back["small"].array.equals(arr)

(True, True)

In [23]:
unpickled = pickle.loads(pickle.dumps(s))
unpickled.equals(s), unpickled.dtype

tmpdir.cleanup()

## Missing elements inside a tensor

Arrow allows individual elements of a tensor to be null. Float tensors read them as NaN. For any other value type there is no missing value, so such input is rejected when the array is built rather than silently widened to float.

In [24]:
float_type = pa.fixed_shape_tensor(pa.float64(), [2])
storage = pa.array([[1.0, None], [3.0, 4.0]], type=float_type.storage_type)
TensorExtensionArray(pa.ExtensionArray.from_storage(float_type, storage))[0]

array([ 1., nan])

In [25]:
int_type = pa.fixed_shape_tensor(pa.int64(), [2])
storage = pa.array([[1, None], [3, 4]], type=int_type.storage_type)
try:
    TensorExtensionArray(pa.ExtensionArray.from_storage(int_type, storage))
except ValueError as e:
    print(e)

Tensors of tensor[int64, (2,)] cannot have missing elements, only whole tensors can be missing. Use a floating point value type to read missing elements as NaN.
